In [1]:
import pandas as pd
import matplotlib.pyplot as plt

### Predictions

In [2]:
import torch
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

import sys
import os
sys.path.append(os.path.abspath("/home1/smaruj/pytorch_akita/"))

# from model import SeqNN
from model_v2_compatible import SeqNN

In [3]:
import numpy as np
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
from bioframe.io.fileops import read_bigwig

In [4]:
import torch
from pyfaidx import Fasta

In [5]:
# alpha genome - akita overlap table
df = pd.read_csv("/scratch1/smaruj/models_comparison_Akita_pytorch/alphagenome/mouse_cell_types/alphagenome_akita_test_overlap_500windows.tsv", sep="\t")

In [6]:
gap_df_path = "/project2/fudenber_735/backup/DNN_HiC/data_mm10/mm10.blacklist.rep.bed"
gap_df = pd.read_csv(gap_df_path, sep="\t", header=None, names=["chrom", "start", "end", "fold"])

In [7]:
FASTA_FILE = "/project2/fudenber_735/genomes/mm10/mm10.fa"

COOL_FILE = "/project2/fudenber_735/GEO/bonev_2017_GSE96107/distiller-0.3.1_mm10/results/coolers/HiC_ncx_NPC_all.mm10.mapq_30.2048.cool"

# --- Load Data ---
genome = Fasta(FASTA_FILE)

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [8]:
import random

def one_hot_encode_sequence(sequence_obj):
    sequence = str(sequence_obj).upper()
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    encoded_sequence = np.array([
        base_to_int.get(base, base_to_int[random.choice("ACGT")]) for base in sequence
    ])

    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return np.expand_dims(one_hot_encoded, axis=0)

In [9]:
import re

def extract_coordinates_from_mseq(mseq_str):
    # Regular expression to match the format: chrom:start-end
    match = re.match(r"(?P<chrom>\w+):(?P<start>\d+)-(?P<end>\d+)", mseq_str)
    
    if match:
        chrom = match.group('chrom')
        start = int(match.group('start'))
        end = int(match.group('end'))
        return chrom, start, end
    else:
        raise ValueError(f"Invalid mseq_str format: {mseq_str}")


In [10]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1, bin_size=2048, gaps_df=None):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    chrom, start, end = extract_coordinates_from_mseq(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
    
    ###########
    # Mask for rows/columns full of NaNs
    row_nan_mask = np.all(seq_hic_nan, axis=1)  # Rows with all NaNs
    col_nan_mask = np.all(seq_hic_nan, axis=0)  # Columns with all NaNs
    
    true_row_indices = np.where(row_nan_mask)[0]
    print(f"Indices of rows with NaNs: {true_row_indices}")
    
    # Apply the NaN mask earlier in the process to avoid processing NaN-only rows/columns
    seq_hic_raw[row_nan_mask, :] = np.nan  # Mask entire rows
    seq_hic_raw[:, col_nan_mask] = np.nan  # Mask entire columns
    
    # Check for NaN filtering percentage
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    ###########
    
    # Mask for regions overlapping with gaps
    if gaps_df is not None:
        # Filter gaps_df for the current chromosome
        gaps_chr = gaps_df[gaps_df['chrom'] == chrom]
        
        # Iterate through each gap region and mark the corresponding rows and columns as NaN
        for _, gap in gaps_chr.iterrows():
            gap_start = gap['start']
            gap_end = gap['end']
            
            # Check if the gap overlaps with the current region
            if (gap_start < end) and (gap_end > start):
                # Mark rows and columns that fall within the gap range as NaN
                gap_start_idx = max(gap_start - start, 0) // bin_size  # Avoid negative indices
                gap_end_idx = min(gap_end - start, seq_hic_raw.shape[0]) // bin_size # Avoid out of bounds
                
                # Add the affected rows and columns to the NaN mask
                row_nan_mask[gap_start_idx:gap_end_idx] = True
                col_nan_mask[gap_start_idx:gap_end_idx] = True
                
        # Apply the updated NaN mask for gaps
        seq_hic_raw[row_nan_mask, :] = np.nan
        seq_hic_raw[:, col_nan_mask] = np.nan
    
        true_row_indices = np.where(row_nan_mask)[0]
        print(f"Indices of rows with NaNs: {true_row_indices}")
    
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
        row_nan_mask = row_nan_mask[padding:-padding]
        col_nan_mask = col_nan_mask[padding:-padding]
        
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic

In [11]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector

In [12]:
N = 256
diagonal_offset = 2

In [13]:
import matplotlib.pyplot as plt
import seaborn as sns

In [14]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [15]:
def vector_to_symmetric_matrix(vec, N):
    matrix = torch.zeros((N, N), dtype=vec.dtype)
    triu_indices = torch.triu_indices(N, N)
    matrix[triu_indices[0], triu_indices[1]] = vec
    matrix = matrix + matrix.T - torch.diag(torch.diag(matrix))
    return matrix

In [16]:
# Exclude diagonals: 0 and ±1
def get_upper_tri_mask(n, skip_diagonals=2):
    # Create mask with False on excluded diagonals, True elsewhere in upper triangle
    mask = np.triu(np.ones((n, n), dtype=bool), k=skip_diagonals)
    return mask

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [18]:

# Helper function to set diagonal elements to a specific value
def set_diag(matrix, value, k):
    # Explicitly set the diagonal to 'value' (in this case, np.nan) for each k
    rows, cols = matrix.shape
    for i in range(rows):
        if 0 <= i + k < cols:
            matrix[i, i + k] = value


def from_upper_triu(vector_repr, matrix_len, num_diags):
    # Ensure vector_repr is a NumPy array (if it's a PyTorch tensor, convert it)
    if isinstance(vector_repr, torch.Tensor):
        vector_repr = vector_repr.detach().flatten().cpu().numpy()  # Flatten and convert to NumPy array

    # Initialize a zero matrix of shape (matrix_len, matrix_len)
    z = np.zeros((matrix_len, matrix_len))

    # Get the indices for the upper triangular matrix
    triu_tup = np.triu_indices(matrix_len, num_diags)

    # Assign the values from the vector_repr to the upper triangular part of the matrix
    z[triu_tup] = vector_repr

    # Set the diagonals specified by num_diags to np.nan
    for i in range(-num_diags + 1, num_diags):
        set_diag(z, np.nan, i)

    # Ensure the matrix is symmetric
    return z + z.T

In [19]:
model_indices = [i for i in range(8)]

In [20]:
all_preds = []
all_targets = []

for model_index in model_indices:
    print("Predicting for model", model_index)
    
    model = SeqNN()
    model.load_state_dict(torch.load(f"/scratch1/smaruj/Akita_pytorch_models/finetuned/mouse_models/Bonev2017_ncx_NPC/models/Akita_v2_mouse_Bonev2017_ncx_NPC_model{model_index}_finetuned.pth", map_location=device))
    model.to(device)
    model.eval()
    
    fold_df = df[df["type_akita"] == f"fold{model_index}"]
    
    for i, row in enumerate(fold_df.itertuples(index=False)):
        print("index:", i)
        chr, start, end = row.chr, row.start, row.end
        mseq_str = f"{chr}:{start}-{end}"

        # TARGET
        sequence = genome[chr][start:end]
        ohe_sequence = one_hot_encode_sequence(sequence)
        ohe_sequence = torch.tensor(ohe_sequence, dtype=torch.float32)
        
        matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1, bin_size=2048, gaps_df=gap_df)
        
        mask = get_upper_tri_mask(matrix.shape[0])
        target_vec = matrix[mask]
        
        # Akita's PRED
        output_vec = model(ohe_sequence)
        
        # Detach and convert to NumPy
        output_vec = output_vec.squeeze(0).detach().numpy()
        output_vec = output_vec.squeeze(0)
        
        all_targets.append(target_vec)
        all_preds.append(output_vec)
        

Predicting for model 0
index: 0
num_filtered_bins: 18
Indices of rows with NaNs: [  9  19  32  33  34  70 259 260 261 262 263 281 282 283 309 322 431 522]
num_filtered_bins: 18
Indices of rows with NaNs: [  9  19  32  33  34  70 259 260 261 262 263 281 282 283 309 322 431 522]


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.11/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 1
num_filtered_bins: 14
Indices of rows with NaNs: [110 152 153 154 254 255 259 260 261 362 363 392 393 394]
num_filtered_bins: 14
Indices of rows with NaNs: [110 152 153 154 254 255 259 260 261 362 363 392 393 394]


/home1/smaruj/miniconda3/envs/pytorch_cuda11.8/lib/python3.11/site-packages/cooltools/lib/numutils.py:1376: RuntimeWarning: invalid value encountered in divide
  val_cur = ar_cur / armask_cur


index: 2
num_filtered_bins: 26
Indices of rows with NaNs: [ 21  22  23  29  52 158 201 204 297 319 320 321 393 468 472 473 484 571
 572 580 581 583 598 599 600 609]
num_filtered_bins: 26
Indices of rows with NaNs: [ 21  22  23  29  52 158 201 204 297 319 320 321 393 468 472 473 484 571
 572 580 581 583 598 599 600 609]
index: 3
num_filtered_bins: 25
Indices of rows with NaNs: [  0   1  73 148 152 153 164 251 252 260 261 263 278 279 280 289 321 322
 426 427 435 454 455 602 618]
num_filtered_bins: 25
Indices of rows with NaNs: [  0   1  73 148 152 153 164 251 252 260 261 263 278 279 280 289 321 322
 426 427 435 454 455 602 618]
index: 4
num_filtered_bins: 54
Indices of rows with NaNs: [  6  39  55  66  67  78  79  99 146 162 204 209 223 227 235 236 237 238
 239 256 257 258 264 274 303 304 315 316 317 318 334 335 336 356 357 358
 379 421 422 423 428 435 486 487 502 503 504 510 511 516 542 567 598 617]
num_filtered_bins: 54
Indices of rows with NaNs: [  6  39  55  66  67  78  79  99 146 16

In [21]:
from scipy.stats import spearmanr, pearsonr

In [22]:
all_cropped_preds = np.array(all_preds)
all_targets = np.array(all_targets)

# Flatten the arrays
preds_flat = all_cropped_preds.flatten()
targets_flat = all_targets.flatten()

# Create a mask for valid (non-NaN) entries
valid_mask = ~np.isnan(preds_flat) & ~np.isnan(targets_flat)

# Apply mask
preds_valid = preds_flat[valid_mask]
targets_valid = targets_flat[valid_mask]

In [23]:
# Compute metrics
pearR = pearsonr(preds_valid, targets_valid)[0]
spearmanR = spearmanr(preds_valid, targets_valid)[0]
mse = np.mean((targets_valid - preds_valid) ** 2)

In [24]:
print("Pearson R = ", pearR)
print("Spearnman R = ", spearmanR)
print("MSE = ", mse)

Pearson R =  0.7314666565174811
Spearnman R =  0.6689101573970961
MSE =  0.06886698850511154


In [ ]:
print("Pearson R = ", pearR)
print("Spearnman R = ", spearmanR)
print("MSE = ", mse)